# Module 3 — Data Analysis & Visualization with Python
## Hands-On Tutorial for Applied / Computational Materials

**Level:** IIT M.Tech / PhD Applied Materials / Computational Materials  
**Duration:** One 3-hour tutorial + independent exercises  
**Prerequisites:** Modules 1–2

### Learning philosophy

This tutorial is **not** a generic Pandas/Matplotlib course. Every programming operation is connected to a materials-science data-analysis problem.

The workflow developed here is:

\[
\boxed{\text{Acquire data}\rightarrow\text{inspect}\rightarrow\text{clean}
\rightarrow\text{transform}\rightarrow\text{visualize}\rightarrow\text{interpret}}
\]

### Learning objectives

By the end of this tutorial, students should be able to:

- Load experimental/simulation data from CSV and Excel files.
- Understand Pandas Series and DataFrames.
- Inspect, filter, sort, and transform materials datasets.
- Detect and handle missing values.
- Identify simple data-quality problems and outliers.
- Merge multiple datasets using meaningful keys.
- Use `groupby()` for materials-processing comparisons.
- Produce publication-quality scientific plots with Matplotlib.
- Use Seaborn for statistical visualization.
- Generate histograms, scatter plots, heatmaps, pair plots, and correlation plots.
- Select plots based on the scientific question rather than aesthetics.
- Interpret plots quantitatively and physically.
- Build a reproducible materials-data-analysis workflow.


## Tutorial roadmap

| Section | Topic | Materials-science application |
|---|---|---|
| 1 | Python scientific stack | Data-analysis environment |
| 2 | Creating a materials dataset | Experimental/simulation data |
| 3 | DataFrame inspection | Data quality |
| 4 | CSV / Excel I/O | Experimental datasets |
| 5 | Filtering and sorting | Process/property selection |
| 6 | Missing values | Incomplete measurements |
| 7 | Outlier detection | Experimental anomalies |
| 8 | Feature engineering | Derived materials descriptors |
| 9 | GroupBy | Comparing processing conditions |
| 10 | Merge | Combining experiments and descriptors |
| 11 | Histograms | Distributions |
| 12 | Scatter plots | Property relationships |
| 13 | Scientific plotting | Publication-quality figures |
| 14 | Heatmaps | Correlation / matrices |
| 15 | Pair plots | Multivariate exploration |
| 16 | Integrated analysis | Materials informatics workflow |
| 17 | Exercises | Independent practice |
| 18 | Mini-project | Materials-data investigation |


# 1. Import the scientific Python stack

We will use:

- **NumPy** — numerical arrays and mathematical operations
- **Pandas** — tabular data analysis
- **Matplotlib** — scientific visualization
- **Seaborn** — statistical visualization

Later computational-materials work will also use libraries such as SciPy, pymatgen, matminer, scikit-learn, and others.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

rng = np.random.default_rng(42)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Seaborn:", sns.__version__)


# 2. Create a synthetic materials dataset

For this tutorial we will work with a dataset representing materials-processing experiments.

Each row corresponds to one sample.

Variables include:

- `Sample_ID`
- `Alloy`
- `Heat_Treatment`
- `Temperature_K`
- `Time_h`
- `Grain_Size_um`
- `Hardness_HV`
- `Yield_Strength_MPa`
- `Density_g_cm3`
- `Porosity_pct`
- `Carbon_pct`
- `Tensile_Strength_MPa`

The dataset is synthetic but designed to behave like a realistic materials-science dataset.


In [ ]:
n = 120

alloys = rng.choice(["A", "B", "C"], size=n, p=[0.40, 0.35, 0.25])
heat_treatment = rng.choice(
    ["As-quenched", "Tempered", "Annealed"],
    size=n,
    p=[0.35, 0.40, 0.25]
)

temperature = rng.choice([773, 873, 973, 1073], size=n)
time_h = rng.choice([1, 2, 4, 8], size=n)

carbon = np.clip(rng.normal(0.55, 0.12, n), 0.20, 0.90)

grain_size = (
    5
    + 0.006*(temperature - 773)
    + 0.35*np.log1p(time_h)
    + rng.normal(0, 0.6, n)
)

hardness = (
    260
    - 8.0*grain_size
    + 55*carbon
    + rng.normal(0, 8, n)
)

yield_strength = (
    180
    + 0.75*hardness
    + 18*carbon
    + rng.normal(0, 20, n)
)

density = 7.75 + rng.normal(0, 0.04, n)

porosity = np.clip(
    1.5 + rng.normal(0, 0.7, n)
    + 0.4*(heat_treatment == "Annealed"),
    0.05,
    None
)

tensile_strength = (
    yield_strength
    + 100
    - 12*porosity
    + rng.normal(0, 25, n)
)

materials = pd.DataFrame({
    "Sample_ID": [f"S{i:03d}" for i in range(1, n+1)],
    "Alloy": alloys,
    "Heat_Treatment": heat_treatment,
    "Temperature_K": temperature,
    "Time_h": time_h,
    "Grain_Size_um": grain_size,
    "Hardness_HV": hardness,
    "Yield_Strength_MPa": yield_strength,
    "Density_g_cm3": density,
    "Porosity_pct": porosity,
    "Carbon_pct": carbon,
    "Tensile_Strength_MPa": tensile_strength
})

materials.head()


# 3. Understanding the DataFrame

A Pandas DataFrame is a rectangular table.

Think of it as a computational representation of:

\[
\text{samples}\times\text{measured/derived variables}.
\]

The first step in scientific data analysis is **not plotting**. It is understanding what the dataset actually contains.


In [ ]:
print("Shape:", materials.shape)
print("\nColumns:")
print(materials.columns.tolist())

print("\nData types:")
print(materials.dtypes)


In [ ]:
materials.info()


In [ ]:
materials.describe(include="all").T


### Exercise 1 — Dataset inspection

Inspect the DataFrame and answer:

1. How many samples are present?
2. Which columns are numerical?
3. Which columns are categorical?
4. What are the minimum and maximum temperatures?
5. What is the mean hardness?
6. How many heat-treatment categories are present?
7. Which variable has the largest numerical range?


# 4. Reading and writing CSV files

CSV is one of the most common formats for experimental and simulation data.

Typical workflow:

```python
df = pd.read_csv("data.csv")
```

and:

```python
df.to_csv("cleaned_data.csv", index=False)
```

We will create a CSV file so that the complete I/O workflow can be practiced.


In [ ]:
data_dir = Path("module3_data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "materials_experiment.csv"
materials.to_csv(csv_path, index=False)

print("Saved:", csv_path)


In [ ]:
materials_csv = pd.read_csv(csv_path)

print(materials_csv.shape)
materials_csv.head()


## Excel files

Pandas can also read Excel workbooks:

```python
pd.read_excel("materials_data.xlsx")
```

and write them using:

```python
df.to_excel("materials_data.xlsx", index=False)
```

This requires an Excel engine such as `openpyxl`.

If Excel support is unavailable in your environment, use CSV for the tutorial and install `openpyxl` when needed.


In [ ]:
excel_path = data_dir / "materials_experiment.xlsx"

try:
    materials.to_excel(excel_path, index=False)
    materials_excel = pd.read_excel(excel_path)
    print("Excel round-trip successful:", materials_excel.shape)
except Exception as e:
    print("Excel example could not run in this environment.")
    print("Reason:", e)


# 5. Selecting columns and rows

A common scientific task is to extract only the variables relevant to a particular question.

For example:

> How does grain size relate to hardness?

We only need a subset of the full DataFrame.


In [ ]:
grain_hardness = materials[
    ["Sample_ID", "Grain_Size_um", "Hardness_HV"]
]

grain_hardness.head()


In [ ]:
selected = materials.loc[
    materials["Hardness_HV"] > 240,
    ["Sample_ID", "Alloy", "Grain_Size_um", "Hardness_HV"]
]

selected.head(10)


## Filtering using multiple conditions

Suppose we want samples that satisfy:

\[
T>900\;K
\]

and

\[
H>240\;HV.
\]


In [ ]:
high_temperature_high_hardness = materials[
    (materials["Temperature_K"] > 900)
    & (materials["Hardness_HV"] > 240)
]

high_temperature_high_hardness.head()


### Exercise 2 — Scientific filtering

Find:

1. Samples with grain size below 7 μm.
2. Samples heat-treated above 900 K.
3. Samples with hardness above 250 HV.
4. Samples with hardness above 250 HV **and** porosity below 2%.
5. Samples from alloy B that were tempered.


# 6. Sorting

Sorting is useful for ranking materials or identifying extreme cases.

For example, find the ten hardest samples.


In [ ]:
top_hardness = materials.sort_values(
    "Hardness_HV",
    ascending=False
).head(10)

top_hardness[
    ["Sample_ID", "Alloy", "Heat_Treatment", "Hardness_HV", "Grain_Size_um"]
]


### Exercise 3 — Ranking

Produce:

- five samples with the smallest grain size
- ten samples with the highest yield strength
- five samples with the highest porosity


# 7. Missing values

Real materials datasets frequently contain missing observations because:

- an instrument failed,
- a sample was lost,
- a measurement was below detection limit,
- a test was not performed,
- data were not recorded.

Missing values should be **investigated before being filled or deleted**.


In [ ]:
materials_missing = materials.copy()

missing_indices = rng.choice(materials_missing.index, size=10, replace=False)
materials_missing.loc[missing_indices[:4], "Hardness_HV"] = np.nan
materials_missing.loc[missing_indices[4:7], "Porosity_pct"] = np.nan
materials_missing.loc[missing_indices[7:], "Carbon_pct"] = np.nan

materials_missing.isna().sum()


In [ ]:
missing_fraction = materials_missing.isna().mean() * 100
missing_fraction.sort_values(ascending=False)


## Handling missing values

Common strategies:

### Delete rows

```python
df.dropna()
```

Useful when only a tiny fraction of observations are missing and missingness is approximately random.

### Fill with a statistic

```python
df["x"].fillna(df["x"].median())
```

Median imputation can be more robust to outliers than mean imputation.

### Model-based imputation

For more advanced work, missing values can be estimated using other variables.

**Scientific warning:** never hide missingness simply to make a dataset complete.


In [ ]:
median_hardness = materials_missing["Hardness_HV"].median()

materials_imputed = materials_missing.copy()
materials_imputed["Hardness_HV"] = materials_imputed["Hardness_HV"].fillna(
    median_hardness
)

print("Remaining missing hardness values:",
      materials_imputed["Hardness_HV"].isna().sum())


# 8. Outlier detection

An outlier is an observation that differs substantially from the rest of the data.

It may represent:

- a real unusual material,
- a rare processing condition,
- measurement error,
- data-entry error,
- an unrecognized physical mechanism.

**Never automatically delete an outlier just because it looks inconvenient.**


In [ ]:
Q1 = materials["Hardness_HV"].quantile(0.25)
Q3 = materials["Hardness_HV"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = materials[
    (materials["Hardness_HV"] < lower)
    | (materials["Hardness_HV"] > upper)
]

print("Q1 =", Q1)
print("Q3 =", Q3)
print("IQR =", IQR)
print("Lower bound =", lower)
print("Upper bound =", upper)
print("Number of flagged observations =", len(outliers))


## Visualizing possible outliers

A boxplot summarizes:

- median
- quartiles
- interquartile range
- potential extreme observations


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(y=materials["Hardness_HV"])
plt.ylabel("Hardness (HV)")
plt.title("Hardness distribution and potential outliers")
plt.show()


# 9. Feature engineering

Feature engineering means constructing scientifically meaningful variables from existing data.

For example, define the Hall–Petch-type feature

\[
x=\frac{1}{\sqrt{d}},
\]

where \(d\) is grain size.

A simplified relationship is

\[
\sigma_y=\sigma_0+k_y d^{-1/2}.
\]

This is much more informative than treating grain size as an arbitrary column.


In [ ]:
materials["Inverse_Sqrt_GrainSize"] = (
    1 / np.sqrt(materials["Grain_Size_um"])
)

materials[
    ["Grain_Size_um", "Inverse_Sqrt_GrainSize", "Yield_Strength_MPa"]
].head()


In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(
    materials["Inverse_Sqrt_GrainSize"],
    materials["Yield_Strength_MPa"]
)
plt.xlabel(r"$1/\sqrt{d}$ ($\mu$m$^{-1/2}$)")
plt.ylabel("Yield strength (MPa)")
plt.title("Hall–Petch-type relationship")
plt.grid()
plt.show()


### Exercise 4 — Create derived features

Create:

1. `Strength_to_Density = Yield_Strength_MPa / Density_g_cm3`
2. `Hardness_to_GrainSize = Hardness_HV / Grain_Size_um`
3. `Treatment_Temperature_C = Temperature_K - 273.15`

Explain which derived quantity could be useful for materials selection and why.


# 10. GroupBy — comparing materials-processing conditions

Suppose we want the mean hardness for each heat-treatment condition.

Mathematically:

\[
\bar{H}_{g}
=
\frac{1}{N_g}\sum_{i\in g}H_i.
\]

Pandas makes this straightforward.


In [ ]:
grouped_hardness = (
    materials
    .groupby("Heat_Treatment")["Hardness_HV"]
    .agg(["count", "mean", "std", "min", "max"])
)

grouped_hardness


## Multiple grouping variables

Now compare heat treatment and alloy simultaneously.


In [ ]:
grouped = (
    materials
    .groupby(["Alloy", "Heat_Treatment"])
    .agg(
        Mean_Hardness=("Hardness_HV", "mean"),
        Mean_GrainSize=("Grain_Size_um", "mean"),
        Mean_YieldStrength=("Yield_Strength_MPa", "mean"),
        Samples=("Sample_ID", "count")
    )
    .reset_index()
)

grouped


### Scientific interpretation

`groupby()` is particularly important for materials experiments because it lets us ask:

- Does alloy composition change the response?
- Does heat treatment change the response?
- Is there an interaction between alloy and processing?
- Are apparent differences supported by enough samples?

A difference in group means is **not automatically evidence of causality**. Sample size, uncertainty, confounding variables, and experimental design matter.


# 11. Bar plots for group comparisons

Use bars for quantities such as group means when the comparison itself is the main question.

For distributions, however, scatter/box/violin plots are often more informative.


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=materials,
    x="Heat_Treatment",
    y="Hardness_HV",
    errorbar="sd"
)
plt.xlabel("Heat treatment")
plt.ylabel("Hardness (HV)")
plt.title("Hardness by heat-treatment condition")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


# 12. Histograms

A histogram answers:

> How is a variable distributed?

For example, examine grain-size distribution.


In [ ]:
plt.figure(figsize=(7,4))
plt.hist(
    materials["Grain_Size_um"],
    bins=15,
    edgecolor="black"
)
plt.xlabel("Grain size (µm)")
plt.ylabel("Number of samples")
plt.title("Grain-size distribution")
plt.show()


### Exercise 5 — Distribution analysis

Plot histograms for:

- hardness
- yield strength
- porosity
- carbon content

For each, comment on:

1. central tendency
2. spread
3. skewness
4. possible outliers


# 13. Scatter plots

Scatter plots are appropriate when investigating relationships between two numerical variables.

Example:

\[
d \quad \text{vs.}\quad H.
\]


In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(
    materials["Grain_Size_um"],
    materials["Hardness_HV"],
    alpha=0.75
)
plt.xlabel("Grain size (µm)")
plt.ylabel("Hardness (HV)")
plt.title("Grain size versus hardness")
plt.grid()
plt.show()


## Add a third variable

Color can encode a categorical or continuous variable.

Here, use heat-treatment condition.


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=materials,
    x="Grain_Size_um",
    y="Hardness_HV",
    hue="Heat_Treatment",
    style="Alloy",
    s=70
)
plt.xlabel("Grain size (µm)")
plt.ylabel("Hardness (HV)")
plt.title("Microstructure–property relationship")
plt.grid()
plt.show()


# 14. Scientific plotting with Matplotlib

A scientific figure should make the quantitative message easy to see.

Important components:

- axis labels
- physical units
- sensible limits
- readable ticks
- legend when necessary
- descriptive title/caption
- appropriate scale
- uncertainty information where available

Avoid unnecessary decoration.

### Publication-style example


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

for treatment, group in materials.groupby("Heat_Treatment"):
    ax.scatter(
        group["Grain_Size_um"],
        group["Yield_Strength_MPa"],
        label=treatment,
        alpha=0.75
    )

ax.set_xlabel("Grain size (µm)")
ax.set_ylabel("Yield strength (MPa)")
ax.set_title("Yield strength versus grain size")
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()


## Save a figure

Use a vector format such as PDF or SVG when appropriate for publications.


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(materials["Grain_Size_um"], materials["Hardness_HV"])
ax.set_xlabel("Grain size (µm)")
ax.set_ylabel("Hardness (HV)")
ax.set_title("Hardness versus grain size")
fig.tight_layout()

figure_path = data_dir / "hardness_vs_grain_size.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")

print("Saved:", figure_path)
plt.show()


# 15. Logarithmic scales

Many physical quantities vary over several orders of magnitude.

For example, diffusion coefficient \(D\) can be extremely small.

A logarithmic axis may reveal relationships hidden on a linear scale.


In [ ]:
T = np.linspace(400, 1000, 50)
R = 8.314
D0 = 1e-6
Q = 90000

D = D0*np.exp(-Q/(R*T))

plt.figure(figsize=(7,4))
plt.semilogy(T, D)
plt.xlabel("Temperature (K)")
plt.ylabel("Diffusion coefficient (m²/s)")
plt.title("Arrhenius-type temperature dependence")
plt.grid()
plt.show()


# 16. Heatmaps

A heatmap can represent a matrix using color intensity.

A common materials-informatics application is a correlation matrix.


In [ ]:
numeric_columns = [
    "Temperature_K",
    "Time_h",
    "Grain_Size_um",
    "Hardness_HV",
    "Yield_Strength_MPa",
    "Density_g_cm3",
    "Porosity_pct",
    "Carbon_pct",
    "Tensile_Strength_MPa"
]

corr = materials[numeric_columns].corr()

plt.figure(figsize=(10,8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation matrix of materials variables")
plt.tight_layout()
plt.show()


### Interpretation exercise

Inspect the heatmap.

Identify:

1. strongest positive relationship
2. strongest negative relationship
3. variables with weak linear correlation
4. possible redundant variables

Then ask:

> Does a strong correlation necessarily imply a physical mechanism?

Explain your answer.


# 17. Pair plots

A pair plot gives a quick multivariate overview.

It combines:

- scatter plots for pairs of variables
- distributions along the diagonal
- optional grouping by category


In [ ]:
pair_columns = [
    "Grain_Size_um",
    "Hardness_HV",
    "Yield_Strength_MPa",
    "Porosity_pct",
    "Carbon_pct"
]

sns.pairplot(
    materials[pair_columns + ["Alloy"]],
    hue="Alloy",
    corner=True
)

plt.show()


### When pair plots become problematic

Pair plots can become unreadable for large numbers of variables.

For a high-dimensional materials dataset, use them as an exploratory tool rather than the final scientific figure.


# 18. Distribution comparison with boxplots

Boxplots are useful for comparing distributions across processing conditions.


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(
    data=materials,
    x="Heat_Treatment",
    y="Yield_Strength_MPa",
    hue="Alloy"
)
plt.xlabel("Heat treatment")
plt.ylabel("Yield strength (MPa)")
plt.title("Yield-strength distributions")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


# 19. Error bars and repeated measurements

Suppose each processing condition has repeated measurements.

A mean alone is incomplete. We should also show variation.

For example, calculate:

\[
\text{mean}\pm\text{standard deviation}.
\]


In [ ]:
summary = (
    materials
    .groupby("Heat_Treatment")["Hardness_HV"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

summary["sem"] = summary["std"] / np.sqrt(summary["count"])

summary


In [ ]:
x = np.arange(len(summary))

plt.figure(figsize=(8,5))
plt.errorbar(
    x,
    summary["mean"],
    yerr=summary["std"],
    fmt="o",
    capsize=5
)

plt.xticks(x, summary["Heat_Treatment"], rotation=15)
plt.ylabel("Hardness (HV)")
plt.xlabel("Heat treatment")
plt.title("Mean hardness ± standard deviation")
plt.tight_layout()
plt.show()


# 20. Working with grouped time/temperature data

Pandas can be used to construct tables suitable for plotting.

Calculate mean hardness as a function of processing temperature.


In [ ]:
temperature_summary = (
    materials
    .groupby("Temperature_K")
    .agg(
        Mean_Hardness=("Hardness_HV", "mean"),
        Std_Hardness=("Hardness_HV", "std"),
        N=("Hardness_HV", "count")
    )
    .reset_index()
)

temperature_summary


In [ ]:
plt.figure(figsize=(7,5))

plt.errorbar(
    temperature_summary["Temperature_K"],
    temperature_summary["Mean_Hardness"],
    yerr=temperature_summary["Std_Hardness"],
    fmt="o-",
    capsize=4
)

plt.xlabel("Heat-treatment temperature (K)")
plt.ylabel("Hardness (HV)")
plt.title("Hardness versus treatment temperature")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# 21. Merging datasets

Materials data often come from multiple sources.

For example:

### Experimental table

| Sample_ID | Hardness | Grain size |
|---|---:|---:|

### Composition table

| Sample_ID | Ni | Cr | Mo |
|---|---:|---:|---:|

We need to join them using `Sample_ID`.


In [ ]:
composition = pd.DataFrame({
    "Sample_ID": materials["Sample_ID"],
    "Ni_pct": rng.uniform(8, 12, n),
    "Cr_pct": rng.uniform(16, 20, n),
    "Mo_pct": rng.uniform(0, 3, n)
})

mechanical = materials[
    ["Sample_ID", "Hardness_HV", "Yield_Strength_MPa"]
].copy()

combined = pd.merge(
    mechanical,
    composition,
    on="Sample_ID",
    how="inner"
)

combined.head()


## Merge types

- `inner`: retain matching keys in both datasets
- `left`: retain every row from the left dataset
- `right`: retain every row from the right dataset
- `outer`: retain all keys

Choosing the wrong merge can silently change your dataset size.

Always check the result.


In [ ]:
print("Mechanical rows:", len(mechanical))
print("Composition rows:", len(composition))
print("Combined rows:", len(combined))
print("Unique sample IDs:", combined["Sample_ID"].nunique())


# 22. Data validation after merging

After a merge, check:

- number of rows
- number of unique sample IDs
- missing values
- duplicate keys
- unexpected changes in categorical distributions


In [ ]:
print("Missing values:")
print(combined.isna().sum())

print("\nDuplicate Sample_ID values:")
print(combined["Sample_ID"].duplicated().sum())


# 23. Pivot tables

A pivot table can summarize a multidimensional materials dataset.

For example, calculate mean hardness for alloy × heat-treatment combinations.


In [ ]:
pivot = pd.pivot_table(
    materials,
    values="Hardness_HV",
    index="Alloy",
    columns="Heat_Treatment",
    aggfunc="mean"
)

pivot


In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(pivot, annot=True, fmt=".1f")
plt.title("Mean hardness by alloy and heat treatment")
plt.xlabel("Heat treatment")
plt.ylabel("Alloy")
plt.tight_layout()
plt.show()


# 24. A complete exploratory data-analysis workflow

We now combine the major operations into a reusable workflow.

### Step 1 — Load

### Step 2 — Inspect

### Step 3 — Clean

### Step 4 — Engineer features

### Step 5 — Summarize

### Step 6 — Visualize

### Step 7 — Interpret

The most important principle is that **visualization should answer a scientific question**.


In [ ]:
# Step 1: inspect
df = materials.copy()

print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())

# Step 2: basic summary
summary = df.describe().T
display(summary)

# Step 3: feature engineering
df["Strength_to_Density"] = (
    df["Yield_Strength_MPa"] / df["Density_g_cm3"]
)

# Step 4: group summary
group_summary = (
    df.groupby("Heat_Treatment")
      .agg(
          Mean_Hardness=("Hardness_HV", "mean"),
          Mean_Yield=("Yield_Strength_MPa", "mean"),
          Mean_GrainSize=("Grain_Size_um", "mean"),
          N=("Sample_ID", "count")
      )
)

display(group_summary)


In [ ]:
# Step 5: visualize the main structure of the data

fig, ax = plt.subplots(figsize=(8,5))

for alloy, group in df.groupby("Alloy"):
    ax.scatter(
        group["Grain_Size_um"],
        group["Yield_Strength_MPa"],
        label=f"Alloy {alloy}",
        alpha=0.75
    )

ax.set_xlabel("Grain size (µm)")
ax.set_ylabel("Yield strength (MPa)")
ax.set_title("Exploratory microstructure–property analysis")
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()


# 25. Mini Case Study — Microstructure–Property Relationship

We will investigate the hypothesis:

> **Smaller grain size is associated with higher strength.**

This is related to the Hall–Petch relationship:

\[
\sigma_y=\sigma_0+k_y d^{-1/2}.
\]

### Analysis tasks

1. Plot yield strength against grain size.
2. Plot yield strength against \(1/\sqrt{d}\).
3. Calculate Pearson correlation for both representations.
4. Separate the data by alloy.
5. Estimate a simple linear trend.
6. Inspect residuals.
7. Decide whether the data support the proposed relationship.
8. Discuss limitations of the synthetic dataset and the model.


In [ ]:
x1 = materials["Grain_Size_um"]
x2 = materials["Inverse_Sqrt_GrainSize"]
y = materials["Yield_Strength_MPa"]

r_grain = np.corrcoef(x1, y)[0, 1]
r_hp = np.corrcoef(x2, y)[0, 1]

print("Correlation with grain size =", r_grain)
print("Correlation with 1/sqrt(grain size) =", r_hp)


In [ ]:
# Simple linear fit in Hall-Petch coordinates
slope, intercept = np.polyfit(x2, y, 1)

xfit = np.linspace(x2.min(), x2.max(), 200)
yfit = slope*xfit + intercept

plt.figure(figsize=(7,5))
plt.scatter(x2, y, alpha=0.75, label="Samples")
plt.plot(xfit, yfit, label="Linear trend")
plt.xlabel(r"$1/\sqrt{d}$ ($\mu$m$^{-1/2}$)")
plt.ylabel("Yield strength (MPa)")
plt.title("Hall–Petch-type analysis")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("Estimated intercept =", intercept)
print("Estimated slope =", slope)


# 26. Residual analysis

A fitted model is not enough.

For a model

\[
\hat{y}_i=f(x_i),
\]

the residual is

\[
e_i=y_i-\hat{y}_i.
\]

Residual plots can reveal:

- systematic model errors
- nonlinearity
- changing variance
- outliers
- missing physical variables


In [ ]:
predicted = slope*x2 + intercept
residual = y - predicted

plt.figure(figsize=(7,4))
plt.scatter(predicted, residual)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted yield strength (MPa)")
plt.ylabel("Residual (MPa)")
plt.title("Residual analysis")
plt.grid(alpha=0.25)
plt.show()


# 27. Scientific plotting checklist

Before submitting a figure, ask:

### Axes
- Are variable names clear?
- Are units included?
- Is the scale appropriate?

### Data
- Are all relevant observations shown?
- Are uncertainties represented?
- Are outliers visible rather than silently removed?

### Interpretation
- Does the figure answer a scientific question?
- Is the trend quantitatively supported?
- Could another variable explain the apparent trend?

### Reproducibility
- Is the plotting code included?
- Are preprocessing steps documented?
- Can another researcher regenerate the figure?


# 28. Hands-on exercises

## Exercise A — Grain-size distribution

Using `materials`:

1. Calculate mean, median, standard deviation.
2. Plot a histogram.
3. Plot a boxplot.
4. Identify potential outliers.
5. Explain whether you would remove any observation.

## Exercise B — Heat treatment

Compare hardness for all three heat-treatment conditions.

Produce:
- grouped statistics
- boxplot
- mean ± standard deviation plot

## Exercise C — Alloy comparison

Compare the three alloys in terms of:
- hardness
- yield strength
- grain size

Use at least two appropriate visualization types.

## Exercise D — Correlation analysis

Construct a correlation matrix for all numerical features.

Identify three strong relationships and explain whether each relationship has a plausible materials-science interpretation.

## Exercise E — Data merging

Create a second table containing:
- `Sample_ID`
- measured elastic modulus
- measured fracture toughness

Merge it with the existing dataset and verify that no samples were lost unexpectedly.


# 29. Advanced exercise — Detect a hidden processing effect

Suppose a scatter plot shows a weak overall relationship between grain size and strength.

However, the relationship may be different for each alloy.

### Tasks

1. Calculate the overall correlation.
2. Calculate the correlation separately for each alloy.
3. Plot the data with alloy-specific markers.
4. Fit a line to each alloy.
5. Compare slopes.
6. Explain why aggregation can hide relationships.

This is an introduction to an important materials-informatics concept:

\[
\boxed{\text{population-level trend}\neq\text{subgroup-level trend}}
\]


In [ ]:
overall_corr = materials["Grain_Size_um"].corr(
    materials["Yield_Strength_MPa"]
)

print("Overall correlation:", overall_corr)

for alloy, group in materials.groupby("Alloy"):
    r = group["Grain_Size_um"].corr(group["Yield_Strength_MPa"])
    print(f"Alloy {alloy}: correlation = {r:.3f}")


# 30. Mini-project — Materials Data Exploration

## Objective

Perform a complete exploratory data analysis of a materials dataset.

You may use:

- the dataset from this notebook,
- a CSV dataset from your laboratory,
- a public materials dataset.

### Required sections

#### 1. Scientific question
State a concrete materials-science question.

#### 2. Data description
Explain:
- number of samples
- variables
- units
- data source
- experimental/simulation context

#### 3. Data quality
Report:
- missing values
- duplicates
- suspicious values
- potential outliers

#### 4. Exploratory statistics
Calculate appropriate descriptive statistics.

#### 5. Visualization
Include at least:
- one histogram
- one scatter plot
- one boxplot
- one heatmap

Use a pair plot only if it adds useful information.

#### 6. Feature engineering
Create at least two physically meaningful derived variables.

#### 7. Group comparison
Compare at least two processing/composition groups.

#### 8. Scientific interpretation
Discuss trends, correlations, anomalies, and possible mechanisms.

#### 9. Limitations
Discuss:
- sample size
- measurement uncertainty
- missing variables
- confounding
- extrapolation
- correlation versus causation

#### 10. Reproducibility
The notebook should run from beginning to end without manual modification.


# 31. Suggested assessment rubric

| Component | Marks |
|---|---:|
| Data loading and inspection | 10 |
| Data cleaning and quality analysis | 15 |
| Pandas operations | 15 |
| Statistical analysis | 10 |
| Visualization quality | 15 |
| Feature engineering | 10 |
| Materials-science interpretation | 15 |
| Reproducibility/code quality | 10 |
| **Total** | **100** |


# 32. Module 3 → Module 4 connection

The skills developed here prepare students for the mathematical and machine-learning methods that follow.

### Module 3

\[
\text{Data}
\rightarrow
\text{cleaning}
\rightarrow
\text{statistics}
\rightarrow
\text{visualization}
\]

### Module 4

\[
\text{statistics}
\rightarrow
\text{numerical methods}
\rightarrow
\text{least squares}
\rightarrow
\text{optimization}
\]

### Later modules

\[
\text{clean data}
\rightarrow
\text{features}
\rightarrow
\text{PCA}
\rightarrow
\text{ML}
\rightarrow
\text{materials prediction}
\]

The objective is therefore not simply to learn Pandas or plotting commands.

The real skill is:

\[
\boxed{
\text{materials question}
\rightarrow
\text{data representation}
\rightarrow
\text{quantitative analysis}
\rightarrow
\text{visual evidence}
\rightarrow
\text{physical interpretation}
}
\]


# 33. Key takeaways

By completing this notebook you should be comfortable with:

```python
pd.read_csv()
pd.read_excel()
df.head()
df.info()
df.describe()
df.loc[]
df.iloc[]
df.sort_values()
df.isna()
df.fillna()
df.groupby()
pd.merge()
pd.pivot_table()
```

and with:

```python
plt.plot()
plt.scatter()
plt.hist()
plt.errorbar()
sns.boxplot()
sns.heatmap()
sns.pairplot()
```

More importantly, you should be able to explain **why** each operation is being used in a materials-science workflow.

> **Next step:** Module 4 builds on this foundation by introducing probability, statistics, numerical differentiation/integration, interpolation, root finding, curve fitting, and optimization.
